# Convolutional Neural Network

### Importing the libraries

In [128]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [129]:
tf.__version__

'2.21.0'

In [130]:
# Ubah parameter di sini untuk eksperimen
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32
EPOCHS = 25
FILTERS = 64
KERNEL_SIZE = 5
PRED_IMAGE_PATHS = ['single_prediction/cat_or_dog_1.jpg', 'single_prediction/cat_or_dog_2.jpg']
RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)


## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [131]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('training_set',
                                                 target_size = IMAGE_SIZE,
                                                 batch_size = BATCH_SIZE,
                                                 class_mode = 'binary')

Found 8000 images belonging to 2 classes.


### Preprocessing the Test set

In [132]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('test_set',
                                            target_size = IMAGE_SIZE,
                                            batch_size = BATCH_SIZE,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


## Part 2 - Building the CNN

### Initialising the CNN

In [133]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [134]:
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS, kernel_size=KERNEL_SIZE, activation='relu', input_shape=[IMAGE_SIZE[0], IMAGE_SIZE[1], 3]))


### Step 2 - Pooling

In [135]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [136]:
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS, kernel_size=KERNEL_SIZE, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))


### Step 3 - Flattening

In [137]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [138]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Step 5 - Output Layer

In [139]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [140]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [141]:
history = cnn.fit(x = training_set, validation_data = test_set, epochs = EPOCHS)
train_acc_pct = history.history['accuracy'][-1] * 100
val_acc_pct = history.history['val_accuracy'][-1] * 100
print(f'Final train acc: {train_acc_pct:.2f}%')
print(f'Final val acc: {val_acc_pct:.2f}%')


Epoch 1/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 47ms/step - accuracy: 0.5497 - loss: 0.6874 - val_accuracy: 0.5945 - val_loss: 0.6628
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 46ms/step - accuracy: 0.5931 - loss: 0.6676 - val_accuracy: 0.6115 - val_loss: 0.6464
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 46ms/step - accuracy: 0.6170 - loss: 0.6514 - val_accuracy: 0.6380 - val_loss: 0.6360
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 47ms/step - accuracy: 0.6591 - loss: 0.6224 - val_accuracy: 0.6840 - val_loss: 0.5990
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 21s 48ms/step - accuracy: 0.6802 - loss: 0.5938 - val_accuracy: 0.7165 - val_loss: 0.5837
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 46ms/step - accuracy: 0.7115 - loss: 0.5666 - val_accuracy: 0.7075 - val_loss: 0.5693
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 48ms/step - accuracy: 0.7270 - loss: 0.5446 - val_accuracy: 0.7425 - val_loss: 0.5426
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 49ms/step - accuracy: 0.7340 - loss: 0.5254 - 

## Part 4 - Making a single prediction

In [142]:
import numpy as np
from tensorflow.keras.preprocessing import image

class_indices = training_set.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

print('class_indices:', class_indices)
for pred_path in PRED_IMAGE_PATHS:
    test_image = image.load_img(pred_path, target_size = IMAGE_SIZE)
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis = 0)
    test_image = test_image / 255.0

    result = cnn.predict(test_image, verbose = 0)
    probability = float(result[0][0])
    predicted_index = 1 if probability >= 0.5 else 0
    predicted_label = idx_to_class[predicted_index]

    file_name = pred_path.split('/')[-1]
    print(f"{file_name}: prob={probability:.4f}, pred={predicted_label}")


class_indices: {'cats': 0, 'dogs': 1}
cat_or_dog_1.jpg: prob=1.0000, pred=dogs
cat_or_dog_2.jpg: prob=0.0076, pred=cats


In [143]:
# Dinonaktifkan: output prediksi sekarang sudah dicetak per file di cell sebelumnya
